# Лекция 15 — EDA (разведочный анализ данных): задачи для самостоятельной работы

**Темы:** типы признаков, меры среднего и разброса (mean/median/average, ptp/var/std), пропуски, квартили и выбросы (IQR), корреляция, кросс-таблицы и группировки, вероятностные распределения (биномиальное, равномерное, нормальное) средствами NumPy, гистограммы и cdf, bootstrap и ЦПТ.

- **Задачи 1–10** — простые
- **Задачи 11–20** — средние
- **Задачи 21–30** — сложные

**Все задачи используют датасет `TITANIC`, который создаётся в задаче 1** (мини-выборка из 30 пассажиров). Решения работают без интернета; нужны только `numpy`, `pandas`, `matplotlib`.


## Простые задачи


### 1. Загрузка датасета и первый взгляд
Создайте DataFrame `TITANIC` по коду ниже (это мини-выборка пассажиров, 30 строк, есть пропуски в `age`). Покажите `head()`, `shape` и проверьте размер `(30, 8)`.

```python
import numpy as np
import pandas as pd

TITANIC = pd.DataFrame({
    'pclass':   [3,1,3,1,2,3,1,3,2,2,3,1,3,2,3,1,3,2,3,1,3,2,3,1,2,3,1,3,2,3],
    'sex':      ['male','female','male','male','female','male','female','female',
                 'male','female','male','female','male','female','male','male',
                 'female','male','female','female','male','female','male','female',
                 'male','female','male','female','male','male'],
    'age':      [22,38,26,np.nan,35,35,54,2,27,14,4,58,20,39,55,2,31,35,5,34,
                 19,28,8,33,23,22,17,42,29,3],
    'sibsp':    [1,1,0,1,0,0,0,3,0,1,1,0,0,1,0,1,0,0,4,0,0,0,4,2,0,1,1,0,0,5],
    'parch':    [0,0,0,0,0,0,2,1,2,0,1,0,2,5,0,0,0,0,1,2,0,0,1,0,0,0,0,1,1,2],
    'fare':     [7.25,71.28,7.92,53.1,8.05,8.45,51.86,21.07,11.13,16.7,16.7,
                 26.55,8.05,31.27,16,29.12,18,26,31.38,55.9,7.75,13,29.7,27.75,
                 10.5,7.65,26.55,13,26,27.9],
    'embarked': ['S','C','S','S','S','S','S','S','S','S','S','S','Q','S','S','C',
                 'S','S','Q','S','Q','S','Q','S','S','C','S','S','S','S'],
    'survived': [0,1,1,1,0,0,1,0,1,1,0,1,0,0,1,1,1,0,1,1,1,0,1,1,0,1,0,1,0,0],
})
```

**Подсказка:** этот датасет (`pclass`, `sex`, `age`, `sibsp`, `parch`, `fare`, `embarked`, `survived`) используется во всех задачах дальше.


### 2. Типы данных признаков
Определите, какие колонки числовые, а какие — категориальные: используйте `dtypes` и `select_dtypes`. Проверьте: числовых колонок 6, категориальных — 2 (`sex`, `embarked`).

**Подсказка:** `df.select_dtypes(include='number').columns`.


### 3. Меры среднего
Для колонки `fare` посчитайте `np.mean`, `np.median` и `np.average` с весами `pclass` (чем выше класс, тем меньше вес: `w = 4 - pclass`). Проверьте соотношения и что взвешенное среднее совпадает с ручным счётом `sum(w*x)/sum(w)`.

**Подсказка:** для `age` сначала уберите пропуск (`np.nanmean` или `.dropna()`).


### 4. Меры разброса
Для колонки `fare` посчитайте `np.ptp`, `np.var`, `np.std` и проверьте соотношения: `std**2 == var`, `ptp == max - min`. Сравните `np.std` (генеральная, ddof=0) и `Series.std()` (выборочная, ddof=1) — чем отличаются?

**Подсказка:** np.var/ddof=0 делит на n, pandas-метод — на n−1.


### 5. Частоты категориальных признаков
Посчитайте частоты `sex` и `embarked` через `value_counts()` и через `value_counts(normalize=True)`. Проверьте: женщин 14, мужчин 16, сумма частот = 30.

**Подсказка:** `TITANIC['sex'].value_counts()`.


### 6. Описательные статистики
Вызовите `describe()` для `age` и `fare`. Прочитайте результаты: сколько значений в `age` не пропущено, чему равны квартили Q1 и Q3? Проверьте, что значения в `describe()` совпадают с `np.percentile` для 25/50/75%.

**Подсказка:** `df['age'].describe()['count']`, `np.percentile(age_clean, 25)`.


### 7. Группировка возраста (бинами)
Разбейте `age` на группы bins=[0, 12, 18, 35, 80] через `pd.cut` и посчитайте количество в каждой (`value_counts`). Проверьте: сумма по группам = 29 (пропуск никуда не попал). Посчитайте отдельно число детей (`age <= 12`).

**Подсказка:** `pd.cut(series, bins=[0,12,18,35,80])` возвращает категориальную колонку-диапазоны.


### 8. Пропуски (NaN)
Сколько пропусков в каждой колонке (`isna().sum()`)? Заполните пропуски `age` медианой (`fillna(df['age'].median())`) и проверьте: пропусков стало 0, медиана не изменилась, минимум и максимум не изменились.

**Подсказка:** медиана — устойчивая мера, поэтому заполнение ею не сдвигает центр распределения.


### 9. Генерация нормальной выборки
`np.random.seed(0)`, сгенерируйте 100 000 значений `np.random.normal(loc=5, scale=2)`. Проверьте, что выборочные среднее и стандартное отклонение близки к 5 и 2 (±0.05).

**Подсказка:** `np.random.normal(mean, std, size)`.


### 10. Гистограмма
Постройте гистограмму распределения `fare` (метод `.hist(bins=8)` или `.plot(kind='hist')`) и гистограмму `age`. Визуально ответьте: симметрично ли распределение тарифа? (Это же можно проверить числом: mean сильно больше median → правосторонняя асимметрия.)

**Подсказка:** `TITANIC['fare'].hist(bins=8)` — метод DataFrame для быстрого EDA.


## Средние задачи


### 11. Биномиальное распределение вручную
Без scipy! Посчитайте pmf биномиального распределения по формуле P(X=k) = C(n,k)·pᵏ·qⁿ⁻ᵏ (`math.comb`) для n=10, p=0.3. Проверьте: сумма всех вероятностей = 1, максимум pmf — при k = 3.

**Подсказка:** `from math import comb`; `pmf = comb(n,k) * p**k * (1-p)**(n-k)`.


### 12. Матожидание и дисперсия биномиального
Сгенерируйте 100 000 значений `np.random.binomial(10, 0.3, 100000)` и сравните выборочные среднее и дисперсию с теоретическими: `M = np` и `D = np(1-p)`. Отклонение не больше 0.05.

**Подсказка:** подставьте n=10, p=0.3: M = 3, D = 2.1.


### 13. random.random vs random.rand vs uniform
Продемонстрируйте разницу трёх функций: `np.random.random()` — скаляр [0,1), `np.random.rand(2,3)` — массив заданной формы, `np.random.uniform(-5, 5, 1000)` — равномерное на отрезке. Проверьте формы и границы (min/max).

**Подсказка:** `np.random.uniform` принимает границы, а масштабирование `np.random.random()` меняет и форму вызова.


### 14. Правило трёх сигм
Сгенерируйте 100 000 точек нормального распределения (seed) и проверьте правило 68–95–99.7: какая доля значений попадает в μ±1σ, μ±2σ, μ±3σ? Допуск ±2%.

**Подсказка:** доли считаются как `(np.abs(x - mu) < sigma).mean()`.


### 15. Стандартизация (z-оценки)
Стандартизуйте колонку `fare`: z = (x − mean) / std с `np.std` (ddof=0). Проверьте: у результата среднее ≈ 0, стандартное отклонение ≈ 1. Посчитайте z-score самого дорогого билета (71.28) — сколько сигм от среднего?

**Подсказка:** аргумент `ddof` важен: используйте один и тот же std в числителе и знаменателе.


### 16. Выбросы по правилу IQR
Для `fare` посчитайте Q1, Q3, IQR и границы `[Q1 − 1.5·IQR, Q3 + 1.5·IQR]`. Найдите билеты-выбросы. Проверьте: билет 71.28 помечен как выброс, а билет 26.55 — нет.

**Подсказка:** Q1/Q3 через `np.percentile(fare, [25, 75])`; выбросов у правостороннего распределения обычно больше справа.


### 17. Групповые статистики
Сравните долю выживших по полу: `groupby('sex')['survived'].mean()` (женщины ≈ 0.71, мужчины ≈ 0.44) и медианный возраст по классу `groupby('pclass')['age'].median()`. Проверьте: разница долей выживания больше 0.25.

**Подсказка:** среднее по бинарной колонке `survived` — это и есть доля единиц.


### 18. Корреляция
Посчитайте `TITANIC[['age','fare','pclass','survived']].corr()` и отдельно `np.corrcoef(age_clean, fare_clean)[0, 1]`. Проверьте: матрица симметрична, диагональ = 1, корреляция лежит в [-1, 1]. Есть ли корреляция возраста с выживанием?

**Подсказка:** corr() считает построчные пары и автоматически пропускает NaN — сверяйтесь с очищенными данными вручную.


### 19. Кросс-таблица выживания
Постройте `pd.crosstab(TITANIC['sex'], TITANIC['survived'], margins=True)`: сколько женщин выжило (10), сколько всего выживших (17)? Добавьте нормировку по строкам (`normalize='index'`) и сверьте с задачей 17.

**Подсказка:** `crosstab(строки, столбцы, margins=True, normalize='index')`.


### 20. Boxplot и квартили вручную
Постройте ящик с усами для `fare` по классам (`boxplot(by='pclass')`). Затем посчитайте квартили вручную `np.percentile` и сверьте с `Series.quantile`: медиана boxplot = Q2.

**Подсказка:** `TITANIC.boxplot(column='fare', by='pclass')`.


## Сложные задачи


### 21. EDA-отчёт одной функцией
Напишите `eda_report(df)` — фасад первичного анализа, возвращающий словарь: число строк/столбцов, долю колонок с пропусками, список числовых и категориальных колонок, самую «пропускную» колонку, долю выживших. Проверьте все поля на `TITANIC` (1 из 30 возрастов пропущен → `age`).

**Подсказка:** `df.isna().mean()`, `df.columns[df.isna().any()]`, `df['survived'].mean()`.


### 22. Гистограмма вручную (np.histogram)
Постройте гистограмму `fare` руками через `np.histogram(fare, bins=8, density=True)`: проверьте, что сумма `dens * ширины` = 1 (это плотность), ширина карманов равна `(max-min)/8`, а площадь под «ступенькой» = 1. Сверьте максимум плотности с самой частой группой данных.

**Подсказка:** `np.histogram` возвращает `(counts, edges)`; плотность = counts / (n · ширина).


### 23. Плотность нормального распределения вручную
Реализуйте формулу `pdf(x) = 1/(σ·√(2π)) · exp(−(x−μ)²/(2σ²))` для μ=0, σ=1. Проверьте: pdf(0) = 1/√(2π), pdf чётная, интеграл (сумма pdf × шаг по сетке от −10 до 10) ≈ 1. Сверьте теоретическую кривую с нормированной гистограммой выборки (средняя разница на карманах < 0.05).

**Подсказка:** `np.exp(-0.5*(x/sigma)**2)/(sigma*np.sqrt(2*np.pi))`.


### 24. Эмпирическая функция распределения
Постройте эмпирическую cdf выборки `fare`: отсортируйте значения, `y = np.arange(1, n+1)/n`. Проверьте: y монотонно растёт от ~1/30 до 1. Найдите по cdf медиану (первое x, где y ≥ 0.5) и сверьте с `np.median` — значения могут различаться не больше чем на 2 (шаг эмпирической cdf).

**Подсказка:** `np.argsort` + накопление; обратный поиск — `x[np.searchsorted(y, 0.5)]`.


### 25. Условные вероятности и формула Байеса
По данным посчитайте: P(выжил | женщина), P(выжил | мужчина), P(женщина | выжил). Проверьте, что Байес сходится: `P(F|S) == P(S|F)·P(F)/P(S)` (с точностью до 1e-9, дроби 10/14, 7/16, 10/17).

**Подсказка:** P(A|B) = count(A и B)/count(B).


### 26. Устойчивость медианы
Сравните `fare`: (1) среднее больше медианы (правый хвост); (2) добавьте в копию данных один «супер-билет» за 1000 — медиана сдвинулась не больше чем на 2 рубля, а среднее выросло минимум на 30. Почему медиана — робастная мера?

**Подсказка:** одно экстремальное значение из 31 двигает медиану на полшага ранжирования, а среднее — на x/n.


### 27. Квантили тарифов и выживаемость
Разбейте `fare` на 5 групп по квантилям (`pd.qcut(..., 5, duplicates='drop')`) и посчитайте долю выживших в каждой. Проверьте: доля выживших в самой дорогой группе выше, чем в самой дешёвой.

**Подсказка:** `df.groupby(pd.qcut(df['fare'], 5, duplicates='drop'))['survived'].mean()`.


### 28. Центральная предельная теорема
`np.random.seed(1)`: из популяции `fare` возьмите 2000 выборок по 30 (`np.random.choice(fare, size=(2000, 30))`) и постройте распределение средних. Проверьте: среднее средних ≈ среднее популяции (±3), стандартное отклонение средних ≈ σ/√30 (в пределах ×0.6…×1.5).

**Подсказка:** `samples.mean(axis=1)` — выборочное распределение среднего; CLT работает даже для асимметричной популяции.


### 29. Bootstrap-доверительный интервал
Оцените bootstrap-ом 95% ДИ среднего возраста: 5000 повторных выборок из всех непропущенных значений возраста (`np.random.choice(..., replace=True)`), 2.5-й и 97.5-й перцентили средних. Повторите для выживших и погибших; для каждой группы ширина ДИ больше 1 года, и границы упорядочены.

**Подсказка:** ДИ — это перцентили массива bootstrap-средних, а не самих данных.


### 30. Мини-EDA-исследование
Полный цикл: (1) добавьте признак `family_size = sibsp + parch + 1`; (2) заполните пропуски возраста медианой; (3) посчитайте долю выживших по полу и по размеру семьи (малые семьи ≤2 против больших); (4) постройте 2 гистограммы (age и family_size). Вывод: пол — более сильный предиктор, чем размер семьи: разница долей по полу больше 0.25, а по семье — меньше 0.1.

**Подсказка:** объедините приёмы из задач 8, 17 и 21; вывод оформите `print`-ом со всеми числами.


## Где потренироваться дальше

- [Kaggle: Titanic — машинное обучение от A до Я](https://www.kaggle.com/c/titanic/overview)
- [Pandas: 10 минут](https://pandas.pydata.org/docs/user_guide/10min.html)
- [NumPy for statistics](https://numpy.org/doc/stable/reference/routines.statistics.html)
- [StatQuest: EDA плейлист](https://www.youtube.com/playlist?list=PLblh5JKOoLUSJ8Wh6HOiXuY0F9O0gHYFg)
- [Визуализация в pandas](https://pandas.pydata.org/docs/user_guide/visualization.html)
